# Class 5. Quantum Data Encoding and Feature Maps (Exercises)

EVA: Quantum Machine Learning | ZHAW | Pavel Sulimov

---

Goals of this practice session:

1. Normalize vectors for amplitude encoding.
2. Expand angle-encoded states analytically.
3. Compute and interpret a quantum kernel as an overlap.
4. Build small quantum Gram matrices in Qiskit.
5. Compare simple feature maps, including a re-uploading variant.
6. Cross-check one fidelity computation in PennyLane.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector

import pennylane as qml

---
## Part 1: Math tasks

### M5.1. Amplitude encoding

Given
$$
x = [0.4, 0.2, 0.8, 0.4],
$$
normalize the vector and write the corresponding two-qubit amplitude-encoded state.

In [ ]:
x_amp = np.array([0.4, 0.2, 0.8, 0.4], dtype=float)
norm = ...  # YOUR CODE HERE
psi_amp = ...  # YOUR CODE HERE

print(f"Norm = {norm:.6f}")
print("Normalized amplitudes:", np.round(psi_amp, 6))

### M5.2. Two-qubit angle encoding

Let
$$
x = \left(\frac{\pi}{6}, \frac{\pi}{2}\right).
$$
Compute the state
$$
\lvert\psi(x)\rangle = R_y(x_1)\lvert0\rangle \otimes R_y(x_2)\lvert0\rangle
$$
by expanding it in the computational basis.

In [ ]:
x1 = np.pi / 6
x2 = np.pi / 2

coeffs = np.array([
    ...,
    ...,
    ...,
    ...,
])  # YOUR CODE HERE

print("Coefficients [|00>, |01>, |10>, |11>]:")
print(np.round(coeffs, 6))
print("Norm:", np.linalg.norm(coeffs))

### M5.3. One-qubit quantum kernel

For the one-qubit encoding
$$
\lvert\phi(t)\rangle = R_y(t)\lvert 0\rangle,
$$
compute the quantum kernel for
$$
t = \frac{\pi}{5}, \qquad s = \frac{2\pi}{5}.
$$
Use both the closed-form formula and a direct overlap computation.

In [ ]:
t = np.pi / 5
s = 2 * np.pi / 5

ket_t = np.array([np.cos(t / 2), np.sin(t / 2)], dtype=complex)
ket_s = np.array([np.cos(s / 2), np.sin(s / 2)], dtype=complex)

kernel_formula = ...  # YOUR CODE HERE
kernel_overlap = ...  # YOUR CODE HERE

print(f"Kernel from formula: {kernel_formula:.6f}")
print(f"Kernel from overlap: {kernel_overlap:.6f}")

---
## Part 2: Programming tasks

### P5.1. Build encoded states in Qiskit

Create one basis-encoded state, one amplitude-encoded state, and one two-qubit angle-encoded state. Display their statevectors.

In [ ]:
basis_state = ...  # YOUR CODE HERE
amp_state = ...  # YOUR CODE HERE

qc_angle = QuantumCircuit(2)
qc_angle.ry(0.7, 0)
qc_angle.ry(1.1, 1)
angle_state = ...  # YOUR CODE HERE

print("Basis state |10>:")
print(np.round(basis_state.data, 6))
print()
print("Amplitude state:")
print(np.round(amp_state.data, 6))
print()
print("Angle-encoded state:")
print(np.round(angle_state.data, 6))

### P5.2. Build a small quantum Gram matrix with the adjoint trick

Use one-qubit angle encoding and estimate the symmetric Gram matrix for
`angles = [0.15, 0.9, 1.35, 2.1]`.

Use Qiskit and the probability of measuring `0` after `R_y(y)` followed by `R_y(-x)`.

In [ ]:
sampler = StatevectorSampler()
angles = np.array([0.15, 0.9, 1.35, 2.1])


def adjoint_prob0_exact(x: float, y: float) -> float:
    qc = QuantumCircuit(1)
    qc.ry(y, 0)
    qc.ry(-x, 0)
    return ...  # YOUR CODE HERE

K = np.array([
    [... for y in angles]
    for x in angles
], dtype=float)  # YOUR CODE HERE

print("Quantum Gram matrix:\n", np.round(K, 6))

plt.figure(figsize=(4, 4))
plt.imshow(K, vmin=0.0, vmax=1.0, cmap="viridis")
plt.colorbar(shrink=0.8)
plt.title("One-qubit quantum Gram matrix")
plt.xlabel("j")
plt.ylabel("i")
plt.tight_layout()
plt.show()

### P5.3. Compare a single-pass feature map with re-uploading

Build two one-qubit encodings on the grid `values = np.linspace(0.0, 2.4, 6)`:

1. single pass: `R_y(x)`
2. re-uploading: `R_y(x) -> R_z(0.4x) -> R_y(theta1) -> R_y(x) -> R_z(0.4x) -> R_y(theta2)`

Compare the two kernel matrices.

In [ ]:
def state_single(x: float) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(x, 0)
    return ...  # YOUR CODE HERE


def state_reupload(x: float, theta1: float = 0.5, theta2: float = -0.3) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(x, 0)
    qc.rz(0.4 * x, 0)
    qc.ry(theta1, 0)
    qc.ry(x, 0)
    qc.rz(0.4 * x, 0)
    qc.ry(theta2, 0)
    return ...  # YOUR CODE HERE


def kernel_matrix(values: np.ndarray, state_fn) -> np.ndarray:
    states = [state_fn(v) for v in values]
    return np.array([
        [... for sb in states]
        for sa in states
    ], dtype=float)  # YOUR CODE HERE

values = np.linspace(0.0, 2.4, 6)
K_single = kernel_matrix(values, state_single)
K_reupload = kernel_matrix(values, state_reupload)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, title in zip(
    axes,
    [K_single, K_reupload],
    ["Single-pass", "Re-uploading"],
):
    im = ax.imshow(mat, vmin=0.0, vmax=1.0, cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("j")
    ax.set_ylabel("i")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
plt.tight_layout()
plt.show()

### P5.4. Short PennyLane comparison

Use PennyLane to prepare `R_y(theta)|0>` and verify one pairwise fidelity against the Qiskit result.

In [ ]:
dev = qml.device("default.qubit", wires=1)


@qml.qnode(dev)
def qml_state(theta):
    qml.RY(theta, wires=0)
    return qml.state()

qa = np.array(qml_state(0.9), dtype=complex)
qb = np.array(qml_state(1.4), dtype=complex)
f_qml = ...  # YOUR CODE HERE
f_qiskit = ...  # YOUR CODE HERE

print(f"Qiskit fidelity:    {f_qiskit:.6f}")
print(f"PennyLane fidelity: {f_qml:.6f}")

---
## Summary

Encoding is the mechanism that turns classical vectors into quantum feature vectors. Once the state family is fixed, the kernel matrix is determined by pairwise overlaps, and different encodings induce genuinely different geometries.